## Supply Chain Network Generation

In [1]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path

ROOT = Path("..").resolve().parent  # adjust if needed
with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

sup = pd.read_csv("../data/raw/suppliers.csv")
fac = pd.read_csv("../data/raw/factories.csv")
wh  = pd.read_csv("../data/raw/warehouses.csv")
ret = pd.read_csv("../data/raw/retailers.csv")

sup.head(), fac.head(), wh.head(), ret.head()

(  supplier_id        name   country  base_lead_days  reliability_pct  \
 0          S1   AsiaSteel     China              10             0.95   
 1          S2  GlobalWood  Malaysia               8             0.92   
 
    unit_cost  
 0        5.8  
 1        6.1  ,
   factory_id            name location  capacity_per_week  unit_yield
 0         F1  CentralFactory  Vietnam               5000        0.98,
   wh_id            name region  capacity_units  handling_cost_per_unit
 0    W1  NorthWarehouse     NT            2000                     0.5
 1    W2  SouthWarehouse     WA            3000                     0.4,
   retail_id         name    city region
 0        R1  DarwinStore  Darwin     NT
 1        R2   PerthStore   Perth     WA)

In [2]:
nodes = []
nodes += [{"node_type":"supplier","node_id":r["supplier_id"],"region":r.get("country","NA")} for _,r in sup.iterrows()]
nodes += [{"node_type":"factory","node_id":r["factory_id"],"region":r.get("location","NA")} for _,r in fac.iterrows()]
nodes += [{"node_type":"warehouse","node_id":r["wh_id"],"region":r.get("region","NA")} for _,r in wh.iterrows()]
nodes += [{"node_type":"retailer","node_id":r["retail_id"],"region":r.get("region","NA")} for _,r in ret.iterrows()]
nodes_df = pd.DataFrame(nodes)
nodes_df.head()

,node_type,node_id,region
0,supplier,S1,China
1,supplier,S2,Malaysia
2,factory,F1,Vietnam
3,warehouse,W1,NT
4,warehouse,W2,WA


In [3]:
def rand_cost(low_high):
    low, high = low_high
    return np.round(np.random.uniform(low, high), 2)

def base_lt(default_days):
    return int(default_days)

In [4]:
lanes = []
cost_range = cfg["cost"]["transport_cost_per_unit_range"]

for _, s in sup.iterrows():
    for _, f in fac.iterrows():
        lanes.append({
            "lane_id": f"L_SF_{s['supplier_id']}_{f['factory_id']}",
            "from_type":"supplier", "from_id":s["supplier_id"],
            "to_type":"factory",   "to_id":f["factory_id"],
            "base_lead_days": int(s.get("lead_time_days", 10)),
            "transport_cost_per_unit": rand_cost(cost_range)
        })

len(lanes)

2

In [5]:
lt_fw = cfg["lead_time"]["lane_base_days"]["factory_to_wh"]

for _, f in fac.iterrows():
    for _, w in wh.iterrows():
        lanes.append({
            "lane_id": f"L_FW_{f['factory_id']}_{w['wh_id']}",
            "from_type":"factory", "from_id":f["factory_id"],
            "to_type":"warehouse","to_id":w["wh_id"],
            "base_lead_days": base_lt(lt_fw),
            "transport_cost_per_unit": rand_cost(cost_range)
        })

len(lanes)

4

In [6]:
# Simple region-based mapping (edit as you like)
def choose_primary_wh(ret_region):
    if str(ret_region).upper() in ("NORTH","EAST","NT"):
        return wh.iloc[0]["wh_id"]
    else:
        return wh.iloc[-1]["wh_id"]

lt_wr = cfg["lead_time"]["lane_base_days"]["wh_to_retail"]

for _, r in ret.iterrows():
    primary = choose_primary_wh(r.get("region",""))
    lanes.append({
        "lane_id": f"L_WR_{primary}_{r['retail_id']}",
        "from_type":"warehouse", "from_id": primary,
        "to_type":"retail",     "to_id": r["retail_id"],
        "base_lead_days": base_lt(lt_wr),
        "transport_cost_per_unit": rand_cost(cost_range)
    })

lanes_df = pd.DataFrame(lanes)
lanes_df.head()

,lane_id,from_type,from_id,to_type,to_id,base_lead_days,transport_cost_per_unit
0,L_SF_S1_F1,supplier,S1,factory,F1,10,0.89
1,L_SF_S2_F1,supplier,S2,factory,F1,10,0.49
2,L_FW_F1_W1,factory,F1,warehouse,W1,4,0.64
3,L_FW_F1_W2,factory,F1,warehouse,W2,4,0.80
4,L_WR_W1_R1,warehouse,W1,retail,R1,2,0.53


In [7]:
# Quick checks
assert all(lanes_df["base_lead_days"] > 0)
assert set(lanes_df["from_type"]).issubset({"supplier","factory","warehouse"})
assert set(lanes_df["to_type"]).issubset({"factory","warehouse","retail"})

# Connectivity counts (at least one path S→F, F→W, W→R)
print("SF lanes:", ((lanes_df.from_type=="supplier") & (lanes_df.to_type=="factory")).sum())
print("FW lanes:", ((lanes_df.from_type=="factory") & (lanes_df.to_type=="warehouse")).sum())
print("WR lanes:", ((lanes_df.from_type=="warehouse") & (lanes_df.to_type=="retail")).sum())

# Save
out_dir = Path("../data/processed"); out_dir.mkdir(parents=True, exist_ok=True)
nodes_df.to_csv(out_dir/"nodes_master.csv", index=False)
lanes_df.to_csv(out_dir/"lanes_generated.csv", index=False)
print("✅ Saved nodes_master.csv and lanes_generated.csv")

SF lanes: 2
FW lanes: 2
WR lanes: 2
✅ Saved nodes_master.csv and lanes_generated.csv
